# Neural Network Training with TensorFlow, PyTorch and Logistic Regression

This notebook demonstrates:
1. **Neural Network Training with TensorFlow** - Using Keras on MNIST dataset
2. **Neural Network Training with PyTorch** - Custom neural network with backpropagation
3. **Logistic Regression with TensorFlow** - Binary classification evaluation

All methods show how to import libraries, build models, train, and evaluate performance.

In [6]:
%pip install torch tensorflow keras scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 28.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 44.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [7]:
# 1. Neural Network Training with TensorFlow on MNIST Dataset

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Load MNIST dataset
print("Loading MNIST dataset...")
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data to [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build Sequential model
print("\nBuilding TensorFlow model...")
model_tf = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile model
model_tf.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Train model
print("Training TensorFlow model...")
history = model_tf.fit(x_train, y_train, epochs=5, batch_size=32, verbose=1)

# Evaluate on test data
print("\nEvaluating on test data...")
test_loss, test_accuracy = model_tf.evaluate(x_test, y_test, verbose=0)
print(f"TensorFlow Model - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Loading MNIST dataset...



Building TensorFlow model...
Training TensorFlow model...


/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5


W0000 00:00:1777023748.307573   31467 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9142 - loss: 0.2955
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9588 - loss: 0.1416
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9686 - loss: 0.1046
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9732 - loss: 0.0878
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9761 - loss: 0.0758

Evaluating on test data...


W0000 00:00:1777023789.512951   31467 cpu_allocator_impl.cc:82] Allocation of 31360000 exceeds 10% of free system memory.


TensorFlow Model - Test Loss: 0.0740, Test Accuracy: 0.9762


In [8]:
# 2. Neural Network Training with PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define Neural Network
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Initialize model, loss, and optimizer
print("\nBuilding PyTorch model...")
model_pytorch = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pytorch.parameters(), lr=0.001)

# Prepare data
print("Preparing data...")
x_train_tensor = torch.from_numpy(x_train).float().to(device)
y_train_tensor = torch.from_numpy(y_train).long().to(device)
x_test_tensor = torch.from_numpy(x_test).float().to(device)
y_test_tensor = torch.from_numpy(y_test).long().to(device)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Training loop
print("Training PyTorch model...")
num_epochs = 5
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_pytorch(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Evaluate on test data
print("\nEvaluating on test data...")
model_pytorch.eval()
with torch.no_grad():
    test_outputs = model_pytorch(x_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f"PyTorch Model - Test Accuracy: {accuracy:.4f}")

Using device: cpu

Building PyTorch model...
Preparing data...
Training PyTorch model...


Epoch [1/5], Loss: 0.3209
Epoch [2/5], Loss: 0.1553
Epoch [3/5], Loss: 0.1159
Epoch [4/5], Loss: 0.0959
Epoch [5/5], Loss: 0.0811

Evaluating on test data...
PyTorch Model - Test Accuracy: 0.9761


In [9]:
# 3. Logistic Regression using TensorFlow (Binary Classification)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate binary classification dataset
print("Generating binary classification dataset...")
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                           n_redundant=5, n_classes=2, random_state=42)

# Split into train and test
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X, y, test_size=0.2, random_state=42)

# Build Logistic Regression model (single neuron with sigmoid)
print("\nBuilding Logistic Regression model...")
model_lr = Sequential([
    Dense(1, activation='sigmoid', input_shape=(20,))
])

# Compile with binary crossentropy loss
model_lr.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

# Train the model
print("Training Logistic Regression model...")
history_lr = model_lr.fit(X_train_lr, y_train_lr, epochs=20, batch_size=32, verbose=1)

# Evaluate on test data
print("\nEvaluating on test data...")
loss, accuracy = model_lr.evaluate(X_test_lr, y_test_lr, verbose=0)
print(f"Logistic Regression - Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

# Make predictions
predictions = model_lr.predict(X_test_lr[:5], verbose=0)
print("\nSample Predictions (first 5 test samples):")
for i, pred in enumerate(predictions):
    print(f"Sample {i+1}: {pred[0]:.4f} (Actual: {y_test_lr[i]})")

Generating binary classification dataset...

Building Logistic Regression model...
Training Logistic Regression model...
Epoch 1/20


/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4538 - loss: 2.5141   
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4600 - loss: 2.3269 
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4725 - loss: 2.1519 
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4812 - loss: 1.9868 
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4925 - loss: 1.8329 
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5075 - loss: 1.6919 
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5213 - loss: 1.5598  
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5325 - loss: 1.4420 
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5425 - loss: 1.3297 
Epoch 10/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5537 - loss: 1.2356 
Epoch 11/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5675 - loss: 1.1446 
Epoch 12/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5838 -